# polygon → MOC → shard → 3-D → numpy

The whole zagg read stack in two libraries, one `%pip install`, and zero
credentials. A geojson polygon becomes a morton Multi Order Coverage (MOC); the MOC checks itself
against the store's own coverage; the covered shards open with timings; one
shard renders in 3-D (ATL03 + GEDI together); the current view exports to
voxel cubes on any grid you name and saves to disk.

Everything below is reader-side and calls no zagg public API — `mortie` for
the geometry, `moczarr` for the store, plus the t-digest algebra that
`moczarr[zagg]` imports from zagg. It all runs anonymously
against public S3, binder-ready (or locally for lower latency on the 3D viewer).

Its sibling is [`waveform_viewer.ipynb`](waveform_viewer.ipynb), which takes
the same polygon and stores down to the cell-level join: one GEDI o18
footprint against the 2×2 ATL03 o19 cells beneath it, both rebuilt from their
stored t-digests. The two are separate notebooks because this one needs
`%matplotlib widget` and that one `%matplotlib inline`; the backends collide
in a single kernel.

### Local setup (skip on binder)

`moczarr[zagg]` pulls in `zagg`, which needs **Python ≥ 3.12** — build the
environment on one before installing:

```bash
# conda / mamba
mamba create -n zagg-demo -c conda-forge python=3.12 jupyterlab
mamba activate zagg-demo
pip install mortie "moczarr[zagg]>=0.7" matplotlib ipympl ipywidgets ipyleaflet

# or plain pip, from a 3.12+ interpreter
python3.12 -m venv .venv && . .venv/bin/activate
pip install jupyterlab mortie "moczarr[zagg]>=0.7" matplotlib ipympl ipywidgets ipyleaflet
```

Binder's image already carries all of this; the `%pip` cell below only tops
up an existing kernel.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipympl ipywidgets ipyleaflet
%matplotlib widget


import moczarr as mz
import numpy as np

# The drawing lives in viewers.py and the exports in export.py, both beside this
# notebook, so the cells here stay about the READ path.
from export import registered_pair, voxel_chips
from mortie import decimal_to_word, moc, mort2geo, mort2polygon
from viewers import densest_shard, view3d, viewer_stats

# One store per product, each appendable. Coverage answers which ground the
# store holds; the store name never does.
STORES = {
    "atl03": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr",
        "19/h_tdigest_signal",
    ),
    "gedi": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr",
        "18/rx_flux",
    ),
}
S3 = {"region": "us-west-2", "anonymous": True}

## One polygon in, covered shards out

The polygon below sits on SERC, the mid-Atlantic forest site the HHDC
diffusion papers are built on. Replace it with any area within California or a
NEON AOP site — it need not be a box — and the cell tests whether each store
actually covers what you asked for. An AOI in Alaska will pass
the ATL03 check and fail the GEDI one — GEDI flies on the ISS, so it sees no
higher than |lat| 51.6 and the Alaska NEON sites are outside its reach.

In [ ]:
aoi = {
    "features": [
        {
            "geometry": {
                "coordinates": [
                    [
                        [-76.5750, 38.9000],
                        [-76.5450, 38.9000],
                        [-76.5450, 38.8780],
                        [-76.5600, 38.8720],
                        [-76.5750, 38.8800],
                        [-76.5750, 38.9000],
                    ]
                ]
            }
        }
    ]
}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not contain the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)

# Most GEDI granules, the rule demo/06_paired used. Named once, used below.
SHARD = densest_shard(STORES["gedi"][0], shards, **S3)
print(f"{len(shards)} shards cover the polygon: {shards}\nworking {SHARD}")

## Open one shard — and price a full sweep of one field


In [ ]:
# Open the leaf for each store. `handles` is all the exports need; the sweep
# below is separate, and only the viewer's dropdown labels depend on it.
handles = {n: (mz.open_leaf(root, SHARD, **S3), field) for n, (root, field) in STORES.items()}
stats = viewer_stats(handles)


## The 3-D view — both sensors, exact centroids, time-aware

In [ ]:
view = view3d(handles, SHARD, stats)  # blocks ordered by coincident cells


## Export — voxel cubes, on a grid you choose

Exporting tensors is computationally cheap but not simple: we have to decide what
happens when a chip's data range exceeds the range bins available at the requested
resolution, and how to align range bins between sensors when exporting jointly.

Both are the user's call, i.e. client-side functions. Because the original locations
are stored, a tensor can be built on a 12 m, 1.5 m or 0.38 m spatial grid — down to
1.21 cm at order 29. In the examples below, range bins are isotropic with the
spatial grid by default; the joint export takes whatever z resolution you ask
for, but coarsens it when the two sensors' shared relief will not fit — and
prints that it did.

**1 — ATL03 alone, finer than the store.** Default o22: 1.554 m voxels, z binned to
match, the block emitted as 8×8 isotropic 128³ chips, empty ones skipped. Where a
chip's relief will not fit, `fit_window` trims the tail furthest from the weighted
median first and records the cost — nothing is clipped silently.

In [ ]:
chips, manifest = voxel_chips(handles, view.block)  # the block on screen, at o22
# chips24, _ = voxel_chips(handles, view.block, order=24)  # 0.389 m voxels, 1,024 chips


**2 — ATL03 and GEDI co-registered**, ready to stack: one xy lattice (ATL03's o19,
each GEDI o18 cell replicated into its four children) and one z axis for both.
`read_tensors` derives its window *per sensor*, so two tensors of the same block can
differ in origin *and* in bin height — at which point they share no axis at all. The
cell prints what each sensor would have got alone beside the shared window it uses.


In [ ]:
pair, cubes = registered_pair(handles, view.block)
stacked = np.stack([cubes["atl03"], cubes["gedi"]], axis=0)  # registered, so they stack
print(f"stacked {stacked.shape} — ready for a 2-channel model")


## Read it back — one call

`np.load` returns the cubes dense, in their stored shape, zeros where no
photons landed. Each file also stores the morton cell words of its own grid,
as a plain array in the same (row, col) raster as the cubes — the pair under
the key `"cells"`, the chips as one `<chip id>.cells` array beside each chip
— and `mort2geo`, the inverse of `geo2mort`, turns those words into WGS84
cell-centre `lat`/`lon` grids. z is affine: `z0 + (bin + ½)·dz`, per chip
from the export's `manifest`. Even the filename georeferences itself: its
morton id is a polygon via `mort2polygon`, mapped below.

In [ ]:
dat = np.load(pair)
atl03_counts = dat["atl03"]  # dense (side, side, n_bins) photon counts, 0 = empty voxel
gedi_flux = dat["gedi"]  # same shape, same lattice; flux ~ pe, each o18 footprint spans 2x2 cells
lat, lon = (a.reshape(dat["cells"].shape) for a in mort2geo(dat["cells"].ravel()))
lon -= 360 * (lon > 180)  # mortie keeps lon in [0, 360); N-D input is mortie#219
z = dat["z0"] + (np.arange(atl03_counts.shape[2]) + 0.5) * dat["dz"]
print(
    f"{pair}: atl03 {atl03_counts.shape} ({atl03_counts.sum():,.0f} ph), "
    f"gedi ({gedi_flux.sum():,.0f} pe), lat/lon {lat.shape}, z {z.shape}"
)

In [ ]:
npz = np.load(chips)
name = max(manifest, key=lambda n: manifest[n]["written"])  # the fullest chip
chip, m = npz[name], manifest[name]  # `manifest` is the archive's meta.json, returned at export
lat, lon = (a.reshape(chip.shape[:2]) for a in mort2geo(npz[name + ".cells"].ravel()))
lon -= 360 * (lon > 180)
z = m["z0"] + (np.arange(chip.shape[2]) + 0.5) * m["dz"]
print(f"chip {name}: {chip.shape} {chip.dtype} ({chip.sum():,} ph, kept {m['kept']:.0%})")

In [ ]:
from ipyleaflet import GeoJSON, Map


def footprint(path, zoom=12):
    """The export's ground footprint on a base map, from its filename's morton id."""
    ring = mort2polygon(int(decimal_to_word(path.rsplit("_", 1)[1].removesuffix(".npz"))))
    lats, lons = zip(*ring)  # mort2polygon returns (lat, lon); geojson wants (lon, lat)
    fig = Map(center=(sum(lats) / len(lats), sum(lons) / len(lons)), zoom=zoom)
    fig.add(GeoJSON(data={"type": "Polygon", "coordinates": [[[lo, la] for la, lo in ring]]}))
    return fig


footprint(pair)